# Notebook 3 — Machine Learning Pipeline & 3-Model Comparison
This notebook constructs a PySpark MLlib Pipeline using `VectorAssembler` and `StandardScaler`, comparing **3 Clustering Models** (**K-Means**, **Bisecting K-Means**, and **Gaussian Mixture Models**) evaluated using **Silhouette Score** and **WCSS / Log-Likelihood**.

## SECTION 1 — Initialization & PySpark ML Pipeline

In [5]:
import os
import sys
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans, BisectingKMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Bus Service Benchmarking - ML Pipeline")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Load Parquet Dataset exported from Notebook 1
df = spark.read.parquet("output/benchmark_dataset").cache()

# Select valid active feature columns from benchmark schema
feature_cols = ["Trips", "Stops", "RouteVehicleCount", "ServicePerformanceIndex"]

# Clean feature nulls safely
df_clean = df.fillna(0.0, subset=feature_cols)

# PySpark ML Pipeline Preparation
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)

pipeline_prep = Pipeline(stages=[assembler, scaler])
prep_model = pipeline_prep.fit(df_clean)
ml_data = prep_model.transform(df_clean).cache()

print(f"Prepared {ml_data.count()} Records for Machine Learning Pipeline.")

Prepared 19304 Records for Machine Learning Pipeline.


## SECTION 2 — Model 1: K-Means Clustering

In [6]:
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")

kmeans = KMeans(k=4, seed=42, featuresCol="features", predictionCol="prediction")
km_model = kmeans.fit(ml_data)
km_preds = km_model.transform(ml_data)

km_silhouette = evaluator.evaluate(km_preds)
km_wcss = km_model.summary.trainingCost
print(f"Model 1: K-Means -> Silhouette Score: {km_silhouette:.4f} | WCSS: {km_wcss:.2f}")

Model 1: K-Means -> Silhouette Score: 0.8828 | WCSS: 28754.34


## SECTION 3 — Model 2: Bisecting K-Means Clustering

In [7]:
bkmeans = BisectingKMeans(k=4, seed=42, featuresCol="features", predictionCol="prediction")
bkm_model = bkmeans.fit(ml_data)
bkm_preds = bkm_model.transform(ml_data)

bkm_silhouette = evaluator.evaluate(bkm_preds)
bkm_wcss = bkm_model.summary.trainingCost
print(f"Model 2: Bisecting K-Means -> Silhouette Score: {bkm_silhouette:.4f} | WCSS: {bkm_wcss:.2f}")

Model 2: Bisecting K-Means -> Silhouette Score: 0.4563 | WCSS: 36399.97


## SECTION 4 — Model 3: Gaussian Mixture Model (GMM)

In [8]:
gmm = GaussianMixture(k=4, seed=42, featuresCol="features", predictionCol="prediction")
gmm_model = gmm.fit(ml_data)
gmm_preds = gmm_model.transform(ml_data)

gmm_silhouette = evaluator.evaluate(gmm_preds)
print(f"Model 3: Gaussian Mixture Model -> Silhouette Score: {gmm_silhouette:.4f}")

Model 3: Gaussian Mixture Model -> Silhouette Score: 0.3714


## SECTION 5 — Comparative Model Evaluation & CSV Export

In [9]:
summary_df = pd.DataFrame({
    "Model Algorithm": ["K-Means Clustering", "Bisecting K-Means", "Gaussian Mixture Model"],
    "Silhouette Score": [km_silhouette, bkm_silhouette, gmm_silhouette],
    "WCSS / Loss Metric": [f"{km_wcss:.2f}", f"{bkm_wcss:.2f}", "N/A (Soft Clustering)"]
})

print("\n================ 3-MODEL CLUSTERING COMPARISON ================")
print(summary_df.to_string(index=False))

# Ensure output directory exists
os.makedirs("output", exist_ok=True)

# Export dataset with predictions for Streamlit Dashboard
dashboard_df = km_preds.drop("raw_features", "features")
if dashboard_df.count() > 20000:
    dashboard_df = dashboard_df.limit(20000)

dashboard_df.toPandas().to_csv("output/dashboard_data.csv", index=False)
print("\nExported dashboard data to output/dashboard_data.csv")


================ 3-MODEL CLUSTERING COMPARISON ================
       Model Algorithm  Silhouette Score    WCSS / Loss Metric
    K-Means Clustering          0.882767              28754.34
     Bisecting K-Means          0.456274              36399.97
Gaussian Mixture Model          0.371416 N/A (Soft Clustering)

Exported dashboard data to output/dashboard_data.csv
